# Introdução

## A Biblioteca MNE: Estrutura e Limitações

A MNE é uma biblioteca open-source que suporta o padrão BIDS. Uma limitação técnica importante é que suas ferramentas de visualização interativa funcionam plenamente em ambientes locais (usando backend Qt), mas perdem funcionalidades (como interatividade de janelas) em ambientes de nuvem como o Google Colab.

# Leitura e Visualização de dados brutos, recorte, filtragem e salvamento.

## Importando pacotes que vamos precisar

In [ ]:
import sys # Importa o módulo sys, que fornece acesso a algumas variáveis usadas ou mantidas pelo interpretador Python e a funções que interagem fortemente com o interpretador
!{sys.executable} -m ensurepip --upgrade # Garante que o pip (gerenciador de pacotes do Python) esteja instalado e atualizado no ambiente atual
!{sys.executable} -m pip install --upgrade mne # Usa o pip para instalar (ou atualizar, se já existir) a biblioteca MNE-Python

usage: python -m ensurepip [-h] [--version] [-v] [-U] [--user] [--root ROOT]
                           [--altinstall] [--default-pip]
python -m ensurepip: error: unrecognized arguments: # Garante que o pip (gerenciador de pacotes do Python) esteja instalado e atualizado no ambiente atual


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


In [ ]:
print(mne.__version__) # Imprime a versão do MNE que está instalada no ambiente. É útil para verificar se a instalação foi bem-sucedida

In [3]:
import matplotlib # Para plotagem dos gráficos 
import pathlib # Package que facilita trabalhar com paths no sistema
import mne

Certifique-se de que o Matplotlib utilize o backend `Qt5Agg`, que é a melhor opção para as funções de plotagem interativa do MNE-Python.

In [ ]:
matplotlib.use('Qt5Agg') # Se eu usar isso, todas as figuras que MNE plotar vão estar 'embedded' 

## MNE-Sample-Data

Para os códigos usados nesse curso, usaremos **o dataset de exemplo padrão do própio MNE** que inclui sinais cerebrais brutos, que representam a atividade elétrica ou magnética do cérebro, além de eventos experimentais correspondentes aos estímulos auditivos e visuais apresentados ao participante durante o experimento. 

Também contém informações sobre a localização dos sensores utilizados na aquisição dos sinais, bem como dados básicos do participante. Ademais, o dataset disponibiliza arquivos preparados para a realização de análises completas, incluindo procedimentos de filtragem do sinal, análise por componentes independentes (ICA), estudo de potenciais evocados (ERP/ERF) e análises temporais e espaciais da atividade cerebral.

In [4]:
sample_data_dir = pathlib.Path(mne.datasets.sample.data_path(verbose=True)) # Baixa o dataset de exemplo do MNE e converte para pathlib.Path por conveniência
sample_data_dir


'''

Esse comando executa uma verificação inteligente em duas etapas:

    Verificação: Ele olha no seu computador (geralmente na pasta ~/mne_data) para ver se o dataset "sample" já existe lá.

    Ação:

        Se já existe: Ele não faz nada (não baixa de novo).

        Se NÃO existe: Ele conecta na internet, baixa o arquivo (que é grandinho, cerca de 1.6 GB), descompacta e salva na pasta padrão.
        
Ele retorna o Caminho Absoluto (Path) da pasta onde os dados estão salvos.
'''

C:\Users\Nutes\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FileNotFoundError: Download location c:\Users\Nutes\Documents\EEG Python MNE-20260225T162719Z-3-001\ambiente_mne_eeg\data as specified by MNE_DATA does not exist. Either create this directory manually and try again, or set MNE_DATA to an existing directory.

## Estrutura de Dados (Raw e Info) 

O objeto principal de dados contém não apenas os sinais contínuos, mas também uma estrutura de metadados (Info), que armazena detalhes cruciais como frequência de amostragem, nomes dos canais, data de gravação e canais marcados como ruins.


#### Carregando dados brutos (raw) do dataset!

In [ ]:
raw_path = sample_data_dir / 'MEG' / 'sample' / 'sample_audvis_raw.fif' # .fif é o formato padrão do MNE para armazenar dados EEG/MEG
raw = mne.io.read_raw(raw_path)
raw # Exibe informações sobre os dados carregados, como o número de canais, a taxa de amostragem, a duração da gravação, etc

#### Visualizando 


In [ ]:
raw.plot()

### Dados de MEG, EEG e STI no mesmo plot

- Clicando nos canais, você apaga um pouquinho eles, estamos eliminando os canais ruins
- Podemos adicionar anotações também
  
1. MEG → Magnetoencephalography (sensores magnéticos)
2. EEG → Eletroencefalografia (sensores elétricos)
3. STI → Trigger channels (canais de estímulo)


#### Trigger Chanells

os famosos STI <br>
*STI = STImulus*

- São canais digitais usados para marcar eventos, por exemplo:
    - início de um estímulo visual
    - apresentação de um som
    - botão pressionado
    - marcações geradas pelo equipamento durante a aquisição

Eles não são sinais fisiológicos. Na verdade, são usados para o *event detection*. **Aparecem no plot como uma linha quadrada, com saltos discretos indicando eventos.**

### Butterfly mode 

O **butterfly mode** (modo “borboleta”) é uma forma de visualizar dados no MNE em que todas as curvas dos canais são sobrepostas no mesmo gráfico.

- Pressionamos 'b' pra o butterfly mode
- Imagine que você pega: 306 sinais MEG ou 64 sinais EEG e plota tudo no mesmo painel
- Resultado: parece uma “borboleta” de tantas curvas juntas — por isso o nome.

- Para que serve?
    - Ver se algum canal está com artefato forte
    - Identificar ruído geral
    - Ver padrões globais no sinal
    - Detectar “spikes”, saturação, ou sensores quebrados

### fT/cm

- Uma unidade nesse MEG, o `fT/cm` é femtotesla por centímetro
    - Tesla (T) é a unidade de campo magnético no SI.
    - femto (f) = 10⁻¹⁵
    - Ou seja: 1 fT = 0,000000000000001 tesla
    - É uma unidade extremamente pequena — usada em EEG/MEG, especialmente no MEG, porque os sinais cerebrais magnéticos são muito fracos.
    - /cm = “por centímetro”. Isso significa gradiente, ou seja, variação do campo magnético por centímetro.
    - fT/cm = femtotesla por centímetro. Ou seja, gradientes de campo magnético
<br>
<br>
- Se um sensor tem ruído de 5 fT/√Hz e a blindagem da sala reduz gradientes para 1 fT/cm, isso significa que:
    - o campo magnético varia apenas 1 femtotesla para cada centímetro de distância → ótimo para não contaminar o registro dos campos magnéticos do cérebro.

## Extrair eventos dos canais `STIM`

In [ ]:
events = mne.find_events(raw) # Libera alguns event IDs que não são tão legíveis

# o MNE retorna um array NumPy de shape: (n_eventos, 3), consigo com isso saber os id's dos meus eventos


| **Coluna** | **Nome**                  | **Descrição**                                                                   | **Exemplo** |
| ---------- | ------------------------- | ------------------------------------------------------------------------------- | ----------- |
| **0**      | Amostra (sample)          | Posição no tempo, em número de amostras, onde o evento ocorreu.                 | `1250`      |
| **1**      | Valor anterior do trigger | Estado do canal de estímulo antes da mudança que gerou o evento. Normalmente 0. | `0`         |
| **2**      | ID do evento (event_id)   | Código que identifica o tipo de evento (ex.: estímulo, resposta etc.).          | `32`        |


A coluna 1 representa:

> O valor do canal de estímulo (trigger) antes do evento acontecer.

Ou seja:

- É o estado anterior do canal de eventos.
- Geralmente é 0, porque a maioria dos eventos começa com o canal em repouso.
- Só muda quando o equipamento usa códigos complexos de evento (mistura de bits).


Na maioria dos datasets de EEG, o canal de estímulo sempre volta para 0, então a coluna 1 acaba sendo:

```
0, 0, 0, 0, 0…
```

Por isso você normalmente ignora a coluna 1.

Ela é útil apenas em casos especiais, por exemplo:

MEG que codifica múltiplos eventos com bits misturados

Estímulos muito rápidos sobrepondo triggers

Sistemas que mantêm o trigger alto por vários samples

In [ ]:
## Dicionário para eu armazemar os IDs dos meus eventos

event_id = {
    'Auditory/Left': 1,
    'Auditory/Right': 2,
    'Visual/Left': 3,
    'Visual/Right': 4,
    'Smiley': 5,
    'Button': 32
}
event_id

In [ ]:
events # Apresenta minhas linhas da matrix de evento, que totalizam 320

## Indexação no NumPy

Quando trabalhamos com EEG no Python, muitos dados (como events) são guardados em matrizes NumPy.

Uma matriz é como uma tabela:

Linha | Coluna 0 | Coluna 1 | Coluna 2
-------|-----------|----------|-----------|
0     |   100    |    0     |    1
1     |   250    |    0     |    2
2     |   400    |    0     |    1

No MNE, normalmente:

* Coluna 0 → instante do evento (sample)
* Coluna 2 → tipo do evento (event_id)

**Forma correta de acessar dados no NumPy**
`array[linha, coluna]`

`events[2, 2]` 
*Significa: Pegue o evento da linha 2 e coluna 2 da matriz de eventos. A linha 2 corresponde ao terceiro evento (já que a indexação começa em 0) e a coluna 2 corresponde ao ID do evento. O valor retornado será o ID do evento para esse evento específico, que pode ser usado para identificar o tipo de evento com base no dicionário event_id que criamos anteriormente.*

* O resultado no caso é 1

### Forma que funciona, mas não é recomendada

`events[2][2]`

Essa forma faz duas etapas:
* Pega a linha 2
* Depois pega o elemento 2 dessa linha
Ela funciona, mas é menos eficiente e menos clara.

*Como pegar uma coluna inteira*
`events[:, 0]`
  *Pegue TODAS as linhas da coluna 0*

*Pegar todos os tipos de evento*
`events[:, 2]`

*Como filtrar eventos específicos*

*Exemplo: pegar apenas eventos do tipo 1*
`events[events[:, 2] == 1]`
#Significa: Pegue eventos onde a coluna 2 é igual a 1

*Converter eventos para tempo (segundos)*
`times = events[:, 0] / raw.info['sfreq']`

### SÍNTESE

| Código         | Significado                 |
| -------------- | --------------------------- |
| `events[2, 1]` | linha 2, coluna 1           |
| `events[:, 0]` | todas as linhas da coluna 0 |
| `events[1, :]` | toda a linha 1              |

*Sempre prefira:*
`events[i, j]`

*Evite:*
`events[i][j]`

In [ ]:
len(events[events[:, 2] == 32])


'''
Esta linha conta quantos eventos têm o ID igual a 32, através de uma filtragem detalhada a seguir:

=== PASSO 1: Extrair a coluna de IDs dos eventos ===
events[:, 2] 
- [:] = "todas as linhas"
- [2] = "coluna 2" (terceira coluna, índice começa em 0)
Resultado: um array 1D com todos os valores da coluna 2
Exemplo: [1, 32, 1, 32, 2, 32] -> apenas os IDs dos eventos

=== PASSO 2: Criar uma máscara booleana ===
events[:, 2] == 32
- Compara cada elemento da coluna 2 com o valor 32
- Retorna um array booleano (True/False) com o mesmo tamanho
Exemplo: 
Se events[:, 2] = [1, 32, 1, 32, 2, 32]
Então events[:, 2] == 32 = [False, True, False, True, False, True]

=== PASSO 3: Aplicar a máscara ao array original ===
events[máscara_booleana]
- NumPy usa a máscara para selecionar apenas as linhas onde a máscara é True
- Isso é chamado de "indexação booleana" ou "fancy indexing"
Resultado: um subarray contendo apenas as linhas que passaram no filtro

Exemplo visual:
events = [[100, 0, 1],      máscara = [False]  -> descartada
          [250, 0, 32],                [True]   -> mantida
          [400, 0, 1],                 [False]  -> descartada
          [550, 0, 32]]                [True]   -> mantida

events[events[:, 2] == 32] = [[250, 0, 32],
                               [550, 0, 32]]

=== PASSO 4: Contar as linhas resultantes ===
len(...)
- Retorna o número de linhas que passaram no filtro
- Ou seja, quantas vezes o evento com ID 32 ocorreu

'''

<div class="alert alert-success">
    <b>EXERCÍCIO</b>:
     <ul>
         <li>Quantos eventos <strong>visuais</strong> o dado possui?</li>
    </ul>
</div>

In [ ]:
# <---Resposta possível--->

# ID dos eventos visuais: 3 e 4
len(events[events[:, 2] == 3]) #conta quantos eventos de ID 3 existem na matriz de eventos
len(events[events[:, 2] == 4]) #conta quantos eventos de ID 4 existem na matriz de eventos
total_de_eventos_visuais = len(events[events[:, 2] == 3]) + len(events[events[:, 2] == 4]) #conta o total de eventos visuais (ID 3 + ID 4)
print(total_de_eventos_visuais)

### Por que events[:, 2] == 3 or events[:, 2] == 4 NÃO funciona?

Porque `or` no Python não opera elemento a elemento.
O NumPy tenta converter os arrays booleanos para um único True/False, o que causa:
erro *(ValueError: The truth value of an array is ambiguous)*
ou resultado totalmente errado, dependendo do caso
**NumPy não usa or, and, not.**

#### Forma correta (usando NumPy)

Você precisa usar | para OR elemento a elemento:

```
mask = (events[:, 2] == 3) | (events[:, 2] == 4)
len(events[mask])
```

Ou direto:
```
len(events[(events[:, 2] == 3) | (events[:, 2] == 4)])
```

## Plotando os dados brutos novamente, mas adicionando marcadores de eventos.

In [ ]:
raw.plot(events=events, event_id=event_id) # Agora temos o mesmo EEG mas com a marcação base do nosso dicionário "event_id"

## Informações essenciais sobre os dados

In [ ]:
raw.info # Um monte de informação sobre o nosso sample


### Principais coisas que raw.info contém

| Categoria                       | Exemplos                                              |
| ------------------------------- | ----------------------------------------------------- |
| **Canais**                      | nomes dos canais, tipo (EEG, EOG, EMG, STI, MEG…)     |
| **Sampling rate**               | frequência de amostragem (`sfreq`)                    |
| **Montagem**                    | posição dos eletrodos, sistema 10–20, etc.            |
| **Filtros usados na aquisição** | filtro passa-baixa, passa-alta                        |
| **Informações do equipamento**  | fabricante, sistema Neuromag, Biosemi etc.            |
| **Data de gravação**            | timestamp da sessão                                   |
| **Duração**                     | quantos samples, quantos segundos                     |
| **Bad channels**                | quais canais estão marcados como ruins                |
| **Projetores / compensadores**  | SSP, compensação de ruído etc.                        |
| **Dig points**                  | pontos anatômicos (nasion, tragus etc.), se existirem |


In [ ]:
raw.info['meas_date'] # Ver quando os dados foram gravados

In [ ]:
raw.info['sfreq'] # Obter a frequência de amostragem

In [ ]:
raw.info['bads'] # Saber quais os canais ruins

In [ ]:
raw.ch_names[:10] # Os primeiros 10 canais

In [ ]:
raw.info['chs'][0] # Saber quais tipos de canais existem

In [ ]:
raw.info['dig'] # Ver posição dos eletrodos, se existirem

## Visualizando sensores

In [ ]:
raw.plot_sensors(ch_type='eeg')

- Você consegue até ver o nome dos canais se clicar neles
- Um ponto vermelho indica um canal ruim
- É possível aproximar, mudar posição e tracejado
- É possível salvar a imagem

In [ ]:
raw.plot_sensors(kind='3d', ch_type='eeg')

 Visão 3D, num plano tridimensional, mas sem a cabeça

## Marcando um canal como 'bad'

É possível marcar um canal de eeg como ruim no grafico topográfico (topoplot)

In [ ]:
raw.info['bads'] # Ver os canais marcados como ruins

In [ ]:
# Marcar os canais 'EEG 051' e 'EEG 032' como ruins, adicionando-os à lista de canais ruins em raw.info['bads']
raw.info['bads'] += ['EEG 051', "EEG 032"] 
raw.plot_sensors(ch_type='eeg')

## Selecione apenas um subconjunto dos canais



### **EOG — Eletrooculografia**

Mede Movimentos dos **olhos** (variações do potencial elétrico entre córnea e retina).


#### Onde são colocados os eletrodos?

* Ao lado dos olhos (**movimento horizontal**)
* Acima e abaixo do olho (**movimento vertical**)


#### Para que é usado?

* Detectar **piscadas**, sacadas e movimentos oculares
* Remover artefatos do EEG
* Estudos de atenção e rastreamento ocular
* Avaliação de fadiga e sonolência


#### Características do sinal

* Frequências baixas (0,1–15 Hz)
* Amplitude **muito maior que o EEG**: até **milivolts (mV)**
* Um dos principais artefatos que contaminam o EEG

---

#### **EEG X EOG**

| Característica | EEG                            | EOG                                      |
| -------------- | ------------------------------ | ---------------------------------------- |
| Mede           | Atividade elétrica cerebral    | Movimentos oculares                      |
| Amplitude      | µV (bem pequena)               | mV (bem maior)                           |
| Eletrodos      | Couro cabeludo                 | Em volta dos olhos                       |
| Frequências    | 0,5–40 Hz                      | 0,1–15 Hz                                |
| Usos           | Cognição, sono, epilepsia, BCI | Detecção de piscar e movimento dos olhos |



In [ ]:

# Cria uma cópia do objeto raw original e seleciona apenas os canais de EEG e EOG, deixando de lado os canais de MEG
# O argumento exclude=[] garante que nenhum canal seja excluído do processo de seleção
len(raw_eeg.ch_names) #isso conta o número de canais que foram selecionados na cópia do objeto Raw, ou seja, o número de canais de EEG e EOG presentes em raw_eeg
raw_eeg = raw.copy().pick_types(meg=False, eeg=True, eog=True, exclude=[])

# Obs: Criamos uma cópia do raw porque o pick_types é in-place, ou seja, sobreescreveria os nossos dados originais



O método pick_types seleciona apenas certos tipos de canais do sinal. 

| Argumento    | Significado                         | Resultado                                         |
| ------------ | ----------------------------------- | ------------------------------------------------- |
| `meg=False`  | Não pega canais de **MEG**          | canais MEG são ignorados                          |
| `eeg=True`   | Pega canais **EEG**                 | canais EEG são incluídos                          |
| `eog=True`   | Pega canais **EOG**                 | canais EOG são incluídos                          |
| `exclude=[]` | Não exclui nenhum canal manualmente (os marcados como ruim)| nada é removido além do que os filtros de hardware já tiraram |

Entretanto, ela é considerada ultrapassada, por isso, ao executar a célula é possível se deparar com a seguinte mensagem:
>*NOTE: pick_types() is a legacy function. New code should use inst.pick(...)*

Chamamos uma função de 'Legado' (Legacy) quando não é mais utilizada ou recomendada como ideal 


### Como usar a nova API do MNE (recomendado):

#### O que é um Application Programming Interface (API)?
É o jeito oficial (conjunto de regras, comandos e convenções) que uma biblioteca oferece para você usar as funções dela.

**Analogia**

O MNE é como uma caixa cheia de ferramentas para analisar EEG.

A API é:
* Como você pega e usa essas ferramentas
* Quais comandos são permitidos
* Qual é o jeito "certo" e recomendado de usar

#### Utilizando `inst.pick`
A versão moderna usa inst.pick com seleção por tipo ou nomes:

Exemplo equivalente usando .pick()
`raw_eeg = raw.copy().pick(['eeg', 'eog'])`

| Objetivo                  | Legacy (`pick_types`)            | Novo (`pick`)          |
| ------------------------- | -------------------------------- | ---------------------- |
| Selecionar só EEG         | `pick_types(eeg=True)`           | `pick('eeg')`          |
| Selecionar EEG + EOG      | `pick_types(eeg=True, eog=True)` | `pick(['eeg', 'eog'])` |
| Selecionar tudo menos MEG | `pick_types(meg=False)`          | `pick(['!meg'])`       |


**Vantagens de usar essa nova API**
- Sintaxe mais curta
- Mais consistente com outras partes do MNE
- Suporta filtros avançados (ex.: pick(['eeg', '!bad']))
- Função antiga vai ser removida futuramente

In [ ]:
raw_eeg.info

# agora temos um objeto Raw apenas com os canais de EEG e EOG, e podemos verificar as informações desse novo objeto, como o número de canais, a taxa de amostragem, a duração da gravação, etc.

In [ ]:
raw_eeg.plot(events=events, event_id=event_id)

# Agora temos o mesmo EEG mas apenas com os canais de EEG e EOG, e com a marcação base do nosso dicionário "event_id"

<div class="alert alert-success">
    <b>EXERCÍCIO</b>:
    <ul>
        <li>Selecione apenas os canais MEG ("meg")</li>
        <li>Selecione apenas os canais magnetômetro ("mag")</li>
    </ul>
</div>

In [ ]:
# <---Resposta possível--->

# COM API ANTIGA/LEGACY - MEG
raw_meg = raw.copy().pick_types(meg=True, eeg=False, eog=False, exclude=[])
len(raw_meg.ch_names) # Conta o número de canais que foram selecionados na cópia do objeto raw, ou seja, o número de canais de MEG presentes em raw_meg

# COM A API NOVA - MEG
raw_meg = raw.copy().pick('meg')
len(raw_meg.ch_names)

In [ ]:
# <---Resposta possível--->

raw_meg = raw.copy().pick('mag')
len(raw_meg.ch_names) # Conta o número de canais de magnetômetros presentes em raw_mag, que é a cópia do objeto raw original contendo apenas os canais MAG

## Recortar e filtrar os dados

In [ ]:
raw_eeg_cropped = raw_eeg.copy().crop(tmax=100) # Cria uma cópia do sinal e corta até 100 segundos 
# .copy() é necessário porque crop() modifica o objeto raw original (in-place)
# tmax=100 mantém apenas os dados de 0 a 100 segundos

raw_eeg_cropped.times[-1] # Último elemento do conjunto de pontos de tempos. 'raw_eeg_cropped.times' é um array contendo todos os tempos correspondentes aos samples do sinal


'''
Por que não cheaga exatamente até 100.0?
O sinal é digital (discreto), não contínuo e o último sample depende da taxa de amostragem
Exemplo: com 500 Hz, o intervalo entre amostras é 0.002s
Então o último sample pode ser 99.998s (não exatamente 100.0)

Mais detalhes abaixo⭣
'''

### Taxa de Amostra

Em MNE (e em qualquer sinal digital), **o tempo final quase nunca cai exatamente em 100.000 s**, mesmo que você peça `tmax=100`.
Isso acontece por causa de **como o sinal é amostrado**.



##### O tempo é definido pelos *samples*, não por valores contínuos

Seu EEG não tem tempo contínuo.
Ele tem **pontos discretos**, separados por `1 / fs`, onde `fs` é a frequência de amostragem por SEGUNDO.

Exemplo:

* se `fs = 250 Hz`, cada amostra dura **0,004 s**
* logo os tempos são:
  `0.000, 0.004, 0.008, 0.012, ...`

100 segundos **não é múltiplo exato de 0,004**.
Então **não existe sample exatamente em 100.000s**.

O MNE corta **até o último sample que não ultrapasse 100s**, por exemplo:

* último sample abaixo de 100s: `99.996`
* então `raw.times[-1] = 99.996`


#### Exemplo real

Se:

* `fs = 256 Hz`
* passo temporal = `1 / 256 = 0.00390625 s`

Para chegar exatamente em 100.0, você precisaria de:

```
100 / 0.00390625 = 25600 samples exatos
```

Mas seu sinal pode ter começado em um tempo com arredondamento,
ou sua gravação pode não terminar em um número redondinho de samples.
Daí o último sample pode ser, por exemplo:

```
99.9921875 s
```

---

#### Conclusão

Ele **não chega exatamente em 100s** porque:

1. **O tempo é discreto**, depende da frequência de amostragem.
2. **100s não coincide com a grade temporal dos samples.**
3. O MNE corta no **último ponto antes** de 100s.


In [ ]:
raw_eeg_cropped_filtered = raw_eeg_cropped.filter(l_freq=0.1, h_freq=40) # filter também é in-place

'''
.filter() aplica um filtro passa-banda

Ele mantém apenas as frequências entre 0.1 Hz e 40 Hz:

l_freq=0.1 Hz → remove componentes abaixo de 0.1 Hz
(tendências, drift, variações muito lentas)

h_freq=40 Hz → remove componentes acima de 40 Hz
(ruído muscular, interferência alta, artefatos rápidos)

Isso é padrão em EEG.
'''

> *"By default, MNE does not load data into main memory to conserve resources. inst.filter requires raw data to be loaded. Use preload=True (or string) in the constructor or raw.load_data()."*



Isso aconteceu porque **você tentou aplicar um filtro (`.filter()`) em um objeto `Raw` que ainda *não tinha os dados carregados na memória*.**

 
 #### 1. **MNE não carrega o EEG na Memória RAM automaticamente**

Quando você faz algo como:

```python
raw = mne.io.read_raw_fif("meu_arquivo.fif")
```

Por padrão, **os dados NÃO são carregados na memória RAM**.

Eles ficam **armazenados no disco** e o MNE apenas “aponta” para eles.

Isso economiza memória — especialmente porque arquivos EEG podem ter **vários gigabytes**.

---

#### 2. `.filter()` precisa dos dados completos na RAM

Para filtrar um sinal, o MNE precisa:

* ler todos os samples
* aplicar o filtro FIR/IIR
* escrever o sinal filtrado de volta no objeto

Mas se o dado não está carregado, apenas *pregado no disco*, o MNE não consegue fazer isso.

Por isso ele te obriga a carregar o sinal antes.

---

#### 3. Como carregar os dados?

Existem **duas maneiras**:

**(A) No momento da leitura:**

```python
raw = mne.io.read_raw_fif("arquivo.fif", preload=True)
```

**(B) Depois de ler:**

```python
raw.load_data()
```

Depois disso, o filtro funciona:

```python
raw.filter(l_freq=0.1, h_freq=40)
```

---

#### 4. Por que o MNE faz isso?

Porque muitos pesquisadores trabalham com:

* minutos inteiros de EEG
* centenas de canais
* gravações contínuas
* arquivos de vários GB

Se o MNE carregasse tudo automaticamente, poderia explodir a memória do computador.

Então o comportamento padrão é:

**“Não carregar nada até que o usuário peça explicitamente.”**


In [ ]:
raw_eeg_cropped.load_data() # Agora assim estamos carregando na memória RAM
raw_eeg_cropped_filtered = raw_eeg_cropped.copy().filter(l_freq=0.1, h_freq=40) # Cria uma nova variavel e armazena o eeg filtrado, aplicando um filtro passa-banda aos dados de EEG cortados, mantendo apenas as frequências entre 0.1 Hz e 40 Hz

In [ ]:
# Comparar os sinais brutos x filtrados
raw_eeg_cropped.plot(events=events, event_id=event_id, title = "unfiltered") # bruto
raw_eeg_cropped_filtered.plot(events=events, event_id=event_id, title = "filtered") # filtrado


### PSD

**PSD** significa **Power Spectral Density**, em português:

**Densidade Espectral de Potência**.

É uma forma de representar **como a energia (potência)** de um sinal está distribuída entre **as frequências**.

É uma análise de frequência extremamente usada em EEG.

---

Ela responde a pergunta:

**“Quais frequências têm mais potência no meu sinal?”**

Exemplo para EEG:

* ondas alfa → pico entre **8–12 Hz**
* ondas beta → **13–30 Hz**
* ruído muscular → acima de **30 Hz**
* drift/artefatos lentos → abaixo de **1 Hz**

A PSD mostra tudo isso de forma clara.


---

#### Como a PSD é calculada?

A forma mais comum é com:

* **FFT (Fast Fourier Transform)**
* **Welch’s method** → usado pelo MNE por padrão

O método de Welch divide o sinal em janelas, calcula a FFT de cada uma e tira uma média — por isso é mais estável.

---

#### Interpretação rápida

No gráfico da PSD:

* **Eixo X:** frequência (Hz)
* **Eixo Y:** potência (geralmente em dB ou microvolts²/Hz)

Picos mostram frequências dominantes.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2)

raw_eeg_cropped.plot_psd(ax=ax[0], show=False)
raw_eeg_cropped_filtered.plot_psd(ax=ax[1], show=False)

ax[0].set_title('PSD before filtering')
ax[1].set_title('PSD after filtering')
ax[1].set_xlabel('Frequency (Hz)')
fig.set_tight_layout(True)
plt.show()

**É SUPER IMPORTANTE plotar e visualizar o PSD após aplicar filtros, pra evitar qualquer confusão**

É importante saber **linha por linha**, com clareza, o que esse código faz e **como o MNE + Matplotlib trabalham juntos**.



#### `fig, ax = plt.subplots(2)`

Cria **uma figura** (`fig`) com **2 subplots** (`ax`).

* `ax` será uma lista (na verdade, um array) com duas posições:

  * `ax[0]` → primeiro gráfico
  * `ax[1]` → segundo gráfico
* Eles ficam empilhados verticalmente (2 linhas, 1 coluna)

É onde vamos desenhar os gráficos de PSD.

---

#### `raw_eeg_cropped.plot_psd(ax=ax[0], show=False)`

Chama a função do MNE que plota o PSD do EEG *antes* do filtro.

* A PSD mostra quanta potência existe em cada frequência.
* `ax=ax[0]` → desenha o gráfico **no primeiro subplot**.
* `show=False` → não abre uma janela interativa ainda
  (vamos esperar para mostrar tudo junto no final)

Isso cria o **PSD antes do filtro**.

---

#### `raw_eeg_cropped_filtered.plot_psd(ax=ax[1], show=False)`

Agora plota a PSD **depois do filtro**, no segundo subplot:

* `ax=ax[1]` → segundo gráfico
* Também não mostra ainda

Isso permite comparar o espectro antes/depois do filtro.

---

#### `ax[0].set_title('PSD before filtering')`

Coloca um título no **primeiro** gráfico.

---

#### `ax[1].set_title('PSD after filtering')`

Coloca um título no **segundo** gráfico.

---

#### `ax[1].set_xlabel('Frequency (Hz)')`

Escreve o nome do eixo X **somente no segundo gráfico**
(tipicamente o de baixo).

---

#### `fig.set_tight_layout(True)`

Ajusta espaçamentos automaticamente para evitar:

* títulos cortados
* rótulos sobrepostos
* gráficos espremidos

É recomendável sempre usar.

---

#### `plt.show()`

Mostra os dois gráficos ao mesmo tempo.

Sem isso, os gráficos não apareceriam (a não ser no Jupyter).

---

#### Resumo Conciso

Esse código:

1. Cria dois gráficos empilhados
2. Plota o espectro de frequências **antes do filtro** (em cima)
3. Plota o espectro **depois do filtro** (embaixo)
4. Ajusta títulos e layout
5. Mostra ambos juntos para comparação


<div class="alert alert-success">
    <b>EXERCÍCIO</b>:
    <ul>
        <li>Filtre os dados brutos com um filtro passa-alta de 1 Hz e um filtro passa-baixa de 30 Hz e plote a Densidade Espectral de Potência (PSD).</li>
    </ul>
</div>


In [ ]:
# <---Resposta possível--->

raw_eeg_cropped.load_data()
raw_eeg_cropped_filtered = raw_eeg_cropped.copy().filter(l_freq=1.0, h_freq=30)

fig, ax = plt.subplots(2)

raw_eeg_cropped.plot_psd(ax=ax[0], show=False)
raw_eeg_cropped_filtered.plot_psd(ax=ax[1], show=False)

ax[0].set_title('PSD before filtering')
ax[1].set_title('PSD after filtering')
ax[1].set_xlabel('Frequency (Hz)')
fig.set_tight_layout(True)
plt.show()

## Salvar os dados

In [ ]:
raw_eeg_cropped_filtered.save(pathlib.Path('out_data') / 'eeg_cropped_filt_raw.fif', 
                              overwrite=True) # No caso eu tenho que criar essa pasta out_data, ele não cria por si mesmo

| Trecho                         | Significado                              |
| ------------------------------ | ---------------------------------------- |
| `pathlib.Path('out_data')`     | pasta “out_data”                         |
| `/ 'eeg_cropped_filt_raw.fif'` | juntar a pasta com o nome do arquivo     |
| `overwrite=True`               | permite substituir arquivo existente     |
| `.save(...)`                   | salva o raw no formato padrão MNE (.fif) |
